# MCTS Rollout — Caro 11×11

Notebook thí nghiệm cho tác tử MCTS (tìm kiếm cây Monte Carlo kết hợp rollout ngẫu nhiên).
Mọi thuật toán được import từ `agents/mcts_agent.py`; notebook chỉ tổ chức thí nghiệm và trình bày kết quả.

**Trạng thái hiện tại — Giai đoạn 1:** đã có tầng trạng thái mô phỏng (`MCTSState`), nút cây (`MCTSNode`)
và quy ước giá trị (`outcome_value`). Chưa có vòng tìm kiếm UCT và chưa có tác tử MCTS thi đấu được;
các mục 5–10 sẽ bổ sung ở Giai đoạn 2 và 3.

Chạy notebook bằng **Restart Kernel & Run All** từ kernel `.venv`.

In [1]:
from pathlib import Path
import os

# Chuyển về thư mục Caro để import module chung từ vị trí notebook.
if Path.cwd().name == 'notebooks' and (Path.cwd().parent / 'core').is_dir():
    os.chdir(Path.cwd().parent)
if not (Path.cwd() / 'core' / 'game_engine.py').is_file():
    raise RuntimeError('Hãy mở notebook từ thư mục caro-11x11 hoặc notebooks')

import platform
import sys

import numpy as np
import pandas as pd

from agents.mcts_agent import (DEFAULT_EXPLORATION, DEFAULT_RADIUS, MCTSNode, MCTSState,
                               outcome_value)
from config.search_budget import SearchBudget
from core import CaroGame
from core.game_utils import candidate_actions


def left(df):
    """Căn trái nội dung và tiêu đề bảng để dễ đọc; chỉ dùng để hiển thị."""
    return df.style.set_properties(**{'text-align': 'left'}).set_table_styles(
        [{'selector': 'th', 'props': [('text-align', 'left')]}])

## 1. Câu hỏi nghiên cứu và giới hạn

**Mục tiêu:** xác định câu hỏi mà toàn bộ notebook trả lời và các giới hạn đã biết trước khi đo.

**Câu hỏi nghiên cứu:** MCTS thay đổi chất lượng nước đi và chi phí tính toán như thế nào khi tăng số lượt mô phỏng
(100 / 500 / 1.000 ở chế độ `fixed`) và khi bị giới hạn 2,0 giây/nước (chế độ `time`)?

**Giới hạn đã biết trước khi đo:**
- **Lọc ứng viên:** cây chỉ mở rộng các ô trống trong bán kính 2 quanh quân đã có (`candidate_actions`).
  Đây là phép rút gọn không gian tìm kiếm, không phải tập nước hợp lệ đầy đủ; nước tốt nằm xa hơn sẽ bị bỏ qua.
- **Chính sách rollout:** ngẫu nhiên, không có heuristic chiến thuật. Rollout ngẫu nhiên trên bàn 11×11 thường dài
  và nhiễu, nên MCTS có thể bỏ sót đe dọa ở ngân sách nhỏ. Chính sách cụ thể được chốt và đo ở Giai đoạn 2.
- **Thời gian:** cài đặt bằng Python thuần trên CPU; số mô phỏng đạt được trong 2,0 giây phụ thuộc phần cứng
  và sẽ được ghi nhận thực tế, không giả định trước.

Ô dưới đây đo quy mô của phép lọc ứng viên trên vài thế cờ tạo bằng engine.

**Kỳ vọng trước khi chạy (tính tay):**
- Bàn trống: 121 nước hợp lệ, 1 ứng viên (ô tâm (5, 5) theo quy ước của `candidate_actions`).
- 1 quân ở tâm: 120 nước hợp lệ; ứng viên là ô vuông 5×5 quanh quân trừ ô có quân = 25 − 1 = **24**.
- 1 quân ở góc (0, 0): ô vuông bị cắt ở mép còn 3×3 = 9 ô, trừ ô có quân = **8**.
- 4 quân giữa bàn: các ô vuông 5×5 chồng lên nhau nên ứng viên nhiều hơn 24 nhưng ít hơn 4 × 24.

In [2]:
def play_sequence(moves, size=11, k=5):
    """Tạo thế cờ bằng engine qua chuỗi nước hợp lệ (engine không có API nạp bàn cờ)."""
    game = CaroGame(size, k)
    for move in moves:
        game.step(move)
    return game

positions = {
    'bàn trống': [],
    '1 quân ở tâm': [(5, 5)],
    '4 quân giữa bàn': [(5, 5), (5, 6), (6, 5), (4, 4)],
    '1 quân ở góc': [(0, 0)],
}
rows = []
for name, moves in positions.items():
    game = play_sequence(moves)
    rows.append({'thế cờ': name, 'số quân': game.move_count,
                 'nước hợp lệ': len(game.legal_actions()),
                 f'ứng viên (r={DEFAULT_RADIUS})': len(candidate_actions(game.board, DEFAULT_RADIUS))})
left(pd.DataFrame(rows))

,thế cờ,số quân,nước hợp lệ,ứng viên (r=2)
0,bàn trống,0,121,1
1,1 quân ở tâm,1,120,24
2,4 quân giữa bàn,4,117,40
3,1 quân ở góc,1,120,8


**Quan sát so với kỳ vọng:** số liệu khớp phép tính tay ở cả bốn thế cờ (1 / 24 / 8 / 40).

**Nhận xét:** ở đầu ván, bán kính 2 thu hẹp nhánh từ khoảng 120 nước hợp lệ xuống vài chục ứng viên
(và chỉ 1 ứng viên khi bàn trống). Đây là lý do cây MCTS có thể đi sâu hơn với cùng ngân sách,
nhưng cũng là giới hạn phải công bố khi phân tích các ván thua.

## 2. Môi trường và cấu hình

**Mục tiêu:** ghi toàn bộ cấu hình tại một chỗ để mọi thí nghiệm sau dùng chung và có thể tái lập.

Các tham số dưới đây là cấu hình khởi đầu, chưa phải cấu hình tối ưu thực nghiệm.
Khi một thí nghiệm thay đổi tham số, cấu hình cụ thể sẽ được ghi kèm thí nghiệm đó.

**Định nghĩa thống kê** (chốt trước khi đo, áp dụng từ Giai đoạn 2 trong `agent.stats`, reset mỗi lượt):
| Chỉ số | Cách đếm |
| :--- | :--- |
| `simulations` | Số mô phỏng **hoàn thành**: rollout chạy tới trạng thái kết thúc và đã lan truyền ngược. Mô phỏng bị bỏ dở vì hết thời gian không được đếm và không cập nhật reward. |
| `nodes_visited` | Số nút **mới được tạo** trong cây ở lượt đó (không tính gốc). |
| `pruned_nodes` | Luôn 0: MCTS không cắt tỉa nhánh. Phép lọc ứng viên bán kính 2 không được tính là cắt tỉa. |
| `thinking_time` | Do `BaseAgent.choose_action` đo, đơn vị giây. |

In [3]:
SEED = 42                          # seed cho mọi yếu tố ngẫu nhiên
RADIUS = DEFAULT_RADIUS            # bán kính ứng viên
EXPLORATION = DEFAULT_EXPLORATION  # hệ số khám phá c của UCT (dùng từ Giai đoạn 2)
ROLLOUT_POLICY = 'ngẫu nhiên đều (chốt và đo ở Giai đoạn 2)'
BUDGET_FIXED = SearchBudget(mode='fixed', simulations=1000)
BUDGET_TIME = SearchBudget(mode='time', time_limit_s=2.0)
FIXED_SURVEY = (100, 500, 1000)    # các mức khảo sát ở Giai đoạn 3

def cpu_name():
    """Tên CPU đầy đủ; trên Windows đọc từ registry, nơi khác dùng platform."""
    if sys.platform == 'win32':
        import winreg
        key = winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE, r'HARDWARE\DESCRIPTION\System\CentralProcessor\0')
        return winreg.QueryValueEx(key, 'ProcessorNameString')[0].strip()
    return platform.processor() or platform.machine()

try:
    import torch
    torch_info = f'{torch.__version__} (CUDA: {torch.cuda.is_available()})'
except ImportError:
    torch_info = 'chưa cài'

config = {
    'seed': SEED, 'bán kính ứng viên': RADIUS, 'hệ số khám phá c': round(EXPLORATION, 4),
    'chính sách rollout': ROLLOUT_POLICY,
    'ngân sách fixed': f'{BUDGET_FIXED.simulations} mô phỏng/nước',
    'ngân sách time': f'{BUDGET_TIME.time_limit_s} giây/nước',
    'mức khảo sát fixed': FIXED_SURVEY,
    'Python': sys.version.split()[0], 'NumPy': np.__version__, 'pandas': pd.__version__,
    'PyTorch': torch_info, 'hệ điều hành': platform.platform(),
    'CPU': cpu_name(),
}
left(pd.Series(config, name='giá trị').to_frame())

,giá trị
seed,42
bán kính ứng viên,2
hệ số khám phá c,1.414200
chính sách rollout,ngẫu nhiên đều (chốt và đo ở Giai đoạn 2)
ngân sách fixed,1000 mô phỏng/nước
ngân sách time,2.0 giây/nước
mức khảo sát fixed,"(100, 500, 1000)"
Python,3.11.9
NumPy,2.4.6
pandas,3.1.0


**Nhận xét:** MCTS không dùng PyTorch/GPU; thông tin GPU chỉ ghi lại để đối chiếu môi trường chung của nhóm.
Thời gian chạy của MCTS phụ thuộc CPU nên mọi số liệu thời gian sau này phải đi kèm dòng `CPU` ở trên.

## 3. Kiểm tra trạng thái mô phỏng

**Mục tiêu:** minh họa `MCTSState` trước/sau nước đi và đối chiếu với engine `CaroGame`.

Engine không có API nạp bàn cờ tùy ý hay đi lùi, còn agent chỉ nhận `board` và `player`.
Vì vậy MCTS tự dựng trạng thái bằng `MCTSState.from_board(board, player)` (luôn sao chép bàn cờ)
và đi nước bằng luật giống engine. Tính đúng đắn được kiểm tra bằng cách chạy **cùng chuỗi nước hợp lệ**
trên cả hai và so sánh từng bước. Bộ unit test `tests/test_mcts_agent.py` kiểm tra chặt chẽ việc này;
notebook chỉ minh họa.

**Kỳ vọng trước khi chạy:** ở mọi bước, bàn cờ, người đến lượt, số nước, nước cuối và kết quả của hai bên trùng nhau.
Tại trạng thái kết thúc, `current_player` vẫn là người vừa đi nước cuối (đúng quy ước engine).

In [4]:
SYMBOLS = {0: '.', 1: 'X', -1: 'O'}

def board_text(board):
    """Chuỗi hiển thị bàn cờ; chỉ để minh họa, không thuộc thuật toán."""
    size = board.shape[0]
    head = '   ' + ' '.join(f'{c:>2}' for c in range(size))
    body = [f'{r:>2} ' + ' '.join(f'{SYMBOLS[int(v)]:>2}' for v in row) for r, row in enumerate(board)]
    return '\n'.join([head] + body)

def compare(state, game):
    return {
        'bàn cờ khớp': bool(np.array_equal(state.board, game.board)),
        'lượt khớp': state.current_player == game.current_player,
        'số nước khớp': state.move_count == game.move_count,
        'nước cuối khớp': state.last_action == game.last_action,
        'kết quả khớp': state.result() == game.result(),
    }

# Ván ngắn: X thắng hàng ngang ở nước thứ 9.
moves = [(5, 3), (6, 3), (5, 4), (6, 4), (5, 5), (6, 5), (5, 6), (4, 4), (5, 7)]
game = CaroGame()
state = MCTSState.from_board(game.board, game.current_player)
rows = []
for ply, move in enumerate(moves, start=1):
    game.step(move)
    state.step(move)
    rows.append({'nước': ply, 'tọa độ': move, 'người đi': SYMBOLS[int(state.board[move])],
                 'kết quả (góc X)': 'chưa kết thúc' if state.result() is None else state.result(),
                 'current_player': state.current_player,
                 **compare(state, game)})
trace = pd.DataFrame(rows)
trace

,nước,tọa độ,người đi,kết quả (góc X),current_player,bàn cờ khớp,lượt khớp,số nước khớp,nước cuối khớp,kết quả khớp
0,1,"(5, 3)",X,chưa kết thúc,-1,True,True,True,True,True
1,2,"(6, 3)",O,chưa kết thúc,1,True,True,True,True,True
2,3,"(5, 4)",X,chưa kết thúc,-1,True,True,True,True,True
3,4,"(6, 4)",O,chưa kết thúc,1,True,True,True,True,True
4,5,"(5, 5)",X,chưa kết thúc,-1,True,True,True,True,True
5,6,"(6, 5)",O,chưa kết thúc,1,True,True,True,True,True
6,7,"(5, 6)",X,chưa kết thúc,-1,True,True,True,True,True
7,8,"(4, 4)",O,chưa kết thúc,1,True,True,True,True,True
8,9,"(5, 7)",X,1,1,True,True,True,True,True


In [5]:
check_cols = ['bàn cờ khớp', 'lượt khớp', 'số nước khớp', 'nước cuối khớp', 'kết quả khớp']
print('Mọi bước đều khớp engine:', bool(trace[check_cols].all().all()))
print()
print('Trước nước cuối (đến lượt X):')
before = play_sequence(moves[:-1])
print(board_text(before.board))
print()
print('Sau nước cuối (5, 7) — X nối 5 quân:')
print(board_text(state.board))

Mọi bước đều khớp engine: True

Trước nước cuối (đến lượt X):
    0  1  2  3  4  5  6  7  8  9 10
 0  .  .  .  .  .  .  .  .  .  .  .
 1  .  .  .  .  .  .  .  .  .  .  .
 2  .  .  .  .  .  .  .  .  .  .  .
 3  .  .  .  .  .  .  .  .  .  .  .
 4  .  .  .  .  O  .  .  .  .  .  .
 5  .  .  .  X  X  X  X  .  .  .  .
 6  .  .  .  O  O  O  .  .  .  .  .
 7  .  .  .  .  .  .  .  .  .  .  .
 8  .  .  .  .  .  .  .  .  .  .  .
 9  .  .  .  .  .  .  .  .  .  .  .
10  .  .  .  .  .  .  .  .  .  .  .

Sau nước cuối (5, 7) — X nối 5 quân:
    0  1  2  3  4  5  6  7  8  9 10
 0  .  .  .  .  .  .  .  .  .  .  .
 1  .  .  .  .  .  .  .  .  .  .  .
 2  .  .  .  .  .  .  .  .  .  .  .
 3  .  .  .  .  .  .  .  .  .  .  .
 4  .  .  .  .  O  .  .  .  .  .  .
 5  .  .  .  X  X  X  X  X  .  .  .
 6  .  .  .  O  O  O  .  .  .  .  .
 7  .  .  .  .  .  .  .  .  .  .  .
 8  .  .  .  .  .  .  .  .  .  .  .
 9  .  .  .  .  .  .  .  .  .  .  .
10  .  .  .  .  .  .  .  .  .  .  .


In [6]:
# Dựng trạng thái từ đầu vào kiểu agent (chỉ có board và player) ở giữa ván.
mid = play_sequence(moves[:4])
original = mid.board.copy()
from_input = MCTSState.from_board(mid.board, mid.current_player)
child = from_input.play((5, 5))
pd.DataFrame([
    {'trạng thái': 'dựng từ board', 'lượt': SYMBOLS[from_input.current_player],
     'số nước': from_input.move_count, 'nước cuối': from_input.last_action, 'kết quả': from_input.result()},
    {'trạng thái': 'sau play((5,5))', 'lượt': SYMBOLS[child.current_player],
     'số nước': child.move_count, 'nước cuối': child.last_action, 'kết quả': child.result()},
]).assign(**{'board đầu vào không đổi': bool(np.array_equal(mid.board, original)),
             'trạng thái cha không đổi': from_input.board[5, 5] == 0})

,trạng thái,lượt,số nước,nước cuối,kết quả,board đầu vào không đổi,trạng thái cha không đổi
0,dựng từ board,X,4,None,None,True,True
1,"sau play((5,5))",O,5,"(5, 5)",None,True,True


**Nhận xét:**
- Trạng thái mô phỏng khớp engine ở mọi bước của ván minh họa, kể cả bước kết thúc.
- Khi dựng từ đầu vào của agent, `last_action` là `None` vì agent không được biết nước cuối;
  do đó `from_board` quét toàn bộ bàn cờ một lần để nhận ra chuỗi thắng hoặc bàn đầy có sẵn.
- `play()` trả về trạng thái mới; trạng thái cha và mảng `board` đầu vào không bị thay đổi.
- `from_board` tin vào `player` do bên gọi đưa vào, không suy lượt từ số quân.

## 4. Quan sát cấu trúc cây

**Mục tiêu:** trình bày nút gốc, các nút con, lượt thăm (N), giá trị (Q) và quy ước góc nhìn trên một thế cờ nhỏ.

**Phân định dữ liệu: luật cờ và tìm kiếm**

| Lớp | Trường | Thuộc về | Ý nghĩa |
| :--- | :--- | :--- | :--- |
| `MCTSState` | `board` | Luật cờ | Bản sao bàn cờ `int8` (0 trống, 1 X, -1 O) |
| `MCTSState` | `current_player` | Luật cờ | Người đến lượt; tại terminal là người vừa đi (như engine) |
| `MCTSState` | `last_action`, `move_count` | Luật cờ | Nước cuối (None nếu dựng từ bàn cờ) và số quân đã đi |
| `MCTSState` | `result()` | Luật cờ | None / 1 / -1 / 0 theo góc nhìn X |
| `MCTSNode` | `state` | Liên kết | Trạng thái luật cờ tại nút |
| `MCTSNode` | `parent`, `children` | Tìm kiếm | Liên kết cha/con trong cây |
| `MCTSNode` | `action` | Tìm kiếm | Nước dẫn từ cha tới nút (None ở gốc) |
| `MCTSNode` | `player_just_moved` | Tìm kiếm | Góc nhìn dùng để cộng Q |
| `MCTSNode` | `untried_actions`, `radius` | Tìm kiếm | Ứng viên chưa mở rộng (rỗng nếu terminal) |
| `MCTSNode` | `visits` (N), `value` (Q) | Tìm kiếm | Thống kê, khởi tạo N = 0, Q = 0 |

**Quy ước giá trị (giữ nguyên ở mọi giai đoạn):**
- Kết quả ván `result` theo góc nhìn X như engine: `1` X thắng, `-1` O thắng, `0` hòa.
- Mỗi nút lưu `player_just_moved` — người đã đi nước dẫn tới nút. Ở gốc, đó là đối thủ của người đến lượt.
- Q của nút được cộng dồn theo góc nhìn `player_just_moved`: thắng `+1`, thua `-1`, hòa `0`
  (`outcome_value(result, player) = result × player`).
- Hệ quả: đứng ở nút cha, người đến lượt chính là `player_just_moved` của các nút con,
  nên chọn con có Q/N cao nhất là chọn nước tốt nhất **cho chính mình** — ở tầng của X lẫn tầng của O.
  Không cần đổi dấu khi chọn; dấu được xử lý khi lan truyền ngược (Giai đoạn 2).

**Kỳ vọng trước khi chạy:**
- Bảng ví dụ giá trị: X thắng → X +1, O −1; O thắng → O +1, X −1; hòa → cả hai 0.
- Thế cờ X(5,5), O(5,6), X(4,4), đến lượt O: gốc có `player_just_moved = X`; con tầng 1 là nước của O
  nên `player_just_moved = O`; con tầng 2 là nước của X nên `player_just_moved = X`.
- Mọi nút vừa tạo có N = 0, Q = 0, Q/N = 0 (chưa có tìm kiếm).
- Bàn cờ của gốc và của con tầng 1 không bị thay đổi sau khi mở rộng tầng dưới.

In [7]:
examples = []
for result, label in [(1, 'X thắng'), (-1, 'O thắng'), (0, 'hòa')]:
    examples.append({'kết cục': label, 'result (góc X)': result,
                     'giá trị cho X': outcome_value(result, 1), 'giá trị cho O': outcome_value(result, -1)})
pd.DataFrame(examples)

,kết cục,result (góc X),giá trị cho X,giá trị cho O
0,X thắng,1,1.0,-1.0
1,O thắng,-1,-1.0,1.0
2,hòa,0,0.0,0.0


In [8]:
# Thế cờ nhỏ: X ở (5,5), O ở (5,6), X ở (4,4); đến lượt O. Dùng bán kính 1 để cây dễ quan sát.
small = play_sequence([(5, 5), (5, 6), (4, 4)])
root = MCTSNode.root(small.board, small.current_player, radius=1)
print(board_text(small.board))
print()
print('Người đến lượt ở gốc:', SYMBOLS[root.state.current_player])
print('player_just_moved của gốc:', SYMBOLS[root.player_just_moved])
print('Số ứng viên chưa mở rộng ở gốc:', len(root.untried_actions))

# Mở rộng thủ công 3 nước đầu tiên của gốc và 1 nước ở tầng kế tiếp để quan sát liên kết.
for action in list(root.untried_actions[:3]):
    root.expand(action)
first_child = root.children[next(iter(root.children))]
grandchild = first_child.expand(first_child.untried_actions[0])

def node_row(node, depth):
    return {'độ sâu': depth, 'nước dẫn tới': node.action,
            'player_just_moved': SYMBOLS[node.player_just_moved],
            'đến lượt': SYMBOLS[node.state.current_player] if not node.is_terminal() else '-',
            'N': node.visits, 'Q': node.value, 'Q/N': node.mean_value(),
            'số con': len(node.children), 'chưa mở rộng': len(node.untried_actions),
            'cha': node.parent.action if node.parent and node.parent.parent else ('gốc' if node.parent else '-')}

rows = [node_row(root, 0)]
for child in root.children.values():
    rows.append(node_row(child, 1))
    rows.extend(node_row(g, 2) for g in child.children.values())
pd.DataFrame(rows)

    0  1  2  3  4  5  6  7  8  9 10
 0  .  .  .  .  .  .  .  .  .  .  .
 1  .  .  .  .  .  .  .  .  .  .  .
 2  .  .  .  .  .  .  .  .  .  .  .
 3  .  .  .  .  .  .  .  .  .  .  .
 4  .  .  .  .  X  .  .  .  .  .  .
 5  .  .  .  .  .  X  O  .  .  .  .
 6  .  .  .  .  .  .  .  .  .  .  .
 7  .  .  .  .  .  .  .  .  .  .  .
 8  .  .  .  .  .  .  .  .  .  .  .
 9  .  .  .  .  .  .  .  .  .  .  .
10  .  .  .  .  .  .  .  .  .  .  .

Người đến lượt ở gốc: O
player_just_moved của gốc: X
Số ứng viên chưa mở rộng ở gốc: 14


,độ sâu,nước dẫn tới,player_just_moved,đến lượt,N,Q,Q/N,số con,chưa mở rộng,cha
0,0,None,X,O,0,0.0,0.0,3,11,-
1,1,"(3, 3)",O,X,0,0.0,0.0,1,17,gốc
2,2,"(2, 2)",X,O,0,0.0,0.0,0,22,"(3, 3)"
3,1,"(3, 4)",O,X,0,0.0,0.0,0,16,gốc
4,1,"(3, 5)",O,X,0,0.0,0.0,0,17,gốc


In [9]:
# Tính độc lập: mở rộng con không làm đổi bàn cờ của cha.
print('Bàn cờ gốc vẫn như đầu vào:', bool(np.array_equal(root.state.board, small.board)))
print('Bàn cờ con tầng 1 chưa có nước của tầng 2:',
      int(first_child.state.board[grandchild.action]) == 0)
print('Liên kết cha/con đúng:', grandchild.parent is first_child and first_child.parent is root)

Bàn cờ gốc vẫn như đầu vào: True
Bàn cờ con tầng 1 chưa có nước của tầng 2: True
Liên kết cha/con đúng: True


**Quan sát so với kỳ vọng:** khớp toàn bộ — bảng giá trị đúng như dự kiến, góc nhìn luân phiên X → O → X
theo tầng, mọi nút có N = 0, Q = 0, và ba dòng kiểm tra tính độc lập đều `True`.

**Nhận xét:**
- Gốc có `player_just_moved = X` vì đến lượt O; các con tầng 1 có `player_just_moved = O`,
  các con tầng 2 có `player_just_moved = X` — góc nhìn luân phiên đúng theo tầng.
- Mọi nút mới có N = 0, Q = 0; chưa có số liệu lượt thăm vì vòng tìm kiếm UCT
  (chọn, mở rộng, mô phỏng, lan truyền ngược) chưa được cài đặt.
- Phân bố N/Q thực tế trên cây sẽ được trình bày ở mục 5 sau Giai đoạn 2.

---
*Các mục 5–10 (bốn bước MCTS, năng lực quyết định, khảo sát ngân sách, thi đấu thử, phân tích thất bại, tổng kết)
chưa thực hiện — thuộc Giai đoạn 2 và 3.*